# Customers Bronze — Production ingestion

Supports `INCREMENTAL`, `BACKFILL`, and `FULL_LOAD` for `customers_YYYYMMDD.json` files.

- Explicit source schema is applied to every JSON read.
- Shared global manifest at `metadata/file_manifest/`; filters strictly to `customers` records.
- `INCREMENTAL` skips successful files unless content changed; `BACKFILL` uses date range; `FULL_LOAD` includes successful files.
- Processing writes directly to the dataset's Bronze Delta path. There is no dry-run toggle.
- Use one writer per dataset. Bronze and manifest updates are separate Delta transactions; reconcile if finalization fails after a Bronze write.


In [0]:

# ============================================================
# CELL 1 — CONFIGURATION, EXPLICIT SCHEMA, AND WIDGETS
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    DoubleType
)
from datetime import datetime, timezone, timedelta
import uuid

# ------------------------------------------------------------
# 1. STORAGE AND PATH CONFIGURATION
# ------------------------------------------------------------

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

DATASET_NAME = "customers"
SOURCE_SYSTEM = "customer_api"

LANDING_PATH = BASE_PATH + f"landing/{DATASET_NAME}/"
BRONZE_PATH = BASE_PATH + f"bronze/{DATASET_NAME}/"
MANIFEST_PATH = BASE_PATH + "metadata/file_manifest/"

LEASE_MINUTES = 10

# ------------------------------------------------------------
# 2. EXPLICIT SOURCE SCHEMA
# ------------------------------------------------------------

SOURCE_SCHEMA = StructType([
    StructField("customer_id", LongType(), nullable=False),
    StructField("name", StringType(), nullable=True),
    StructField("city", StringType(), nullable=True),
    StructField("market", StringType(), nullable=True),
])

# ------------------------------------------------------------
# 3. BRONZE OUTPUT COLUMNS
# ------------------------------------------------------------

BRONZE_COLUMNS = [
    "customer_id",
    "name",
    "city",
    "market",
    "ingestion_timestamp",
    "source_system",
    "source_file",
    "run_id",
    "batch_id"
]

# ------------------------------------------------------------
# 4. WIDGETS
# ------------------------------------------------------------

# Remove existing widgets so the notebook can be rerun cleanly.
for name in [
    "run_mode",
    "backfill_start_date",
    "backfill_end_date",
    "reprocess_successful"
]:
    try:
        dbutils.widgets.remove(name)
    except Exception:
        pass

dbutils.widgets.dropdown(
    "run_mode",
    "INCREMENTAL",
    ["INCREMENTAL", "BACKFILL", "FULL_LOAD"],
    "Run Mode"
)

dbutils.widgets.text(
    "backfill_start_date",
    "",
    "Backfill Start Date (YYYY-MM-DD)"
)

dbutils.widgets.text(
    "backfill_end_date",
    "",
    "Backfill End Date (YYYY-MM-DD)"
)

dbutils.widgets.dropdown(
    "reprocess_successful",
    "false",
    ["false", "true"],
    "Reprocess Successful Files?"
)

# ------------------------------------------------------------
# 5. READ AND NORMALIZE PARAMETERS
# ------------------------------------------------------------

raw_run_mode = dbutils.widgets.get("run_mode").strip().upper()

# Accept common aliases from job parameters.
MODE_ALIASES = {
    "INCREMENTAL": "INCREMENTAL",
    "FULL": "FULL_LOAD",
    "FULL_LOAD": "FULL_LOAD",
    "BACKFILL": "BACKFILL"
}

RUN_MODE = MODE_ALIASES.get(raw_run_mode, raw_run_mode)

BACKFILL_START_DATE = (
    dbutils.widgets.get("backfill_start_date").strip()
)

BACKFILL_END_DATE = (
    dbutils.widgets.get("backfill_end_date").strip()
)

REPROCESS_SUCCESSFUL = (
    dbutils.widgets.get("reprocess_successful").strip().lower()
    == "true"
)

# ------------------------------------------------------------
# 6. PARAMETER VALIDATION
# ------------------------------------------------------------

assert RUN_MODE in {
    "INCREMENTAL",
    "BACKFILL",
    "FULL_LOAD"
}, (
    f"Invalid run_mode: {raw_run_mode!r}. "
    "Expected INCREMENTAL, FULL_LOAD, or BACKFILL."
)

if RUN_MODE == "BACKFILL":
    assert (
        BACKFILL_START_DATE and BACKFILL_END_DATE
    ), "BACKFILL requires both dates."

    start_date = datetime.strptime(
        BACKFILL_START_DATE, "%Y-%m-%d"
    ).date()

    end_date = datetime.strptime(
        BACKFILL_END_DATE, "%Y-%m-%d"
    ).date()

    assert (
        start_date <= end_date
    ), "Backfill start date must be <= end date."

# ------------------------------------------------------------
# 7. CONFIGURATION SUMMARY
# ------------------------------------------------------------

print("Dataset:", DATASET_NAME)
print("Raw run mode:", raw_run_mode)
print("Normalized run mode:", RUN_MODE)
print("Reprocess successful:", REPROCESS_SUCCESSFUL)
print("Backfill start date:", BACKFILL_START_DATE or "(not set)")
print("Backfill end date:", BACKFILL_END_DATE or "(not set)")
print("Landing:", LANDING_PATH)
print("Bronze:", BRONZE_PATH)
print("Manifest:", MANIFEST_PATH)


Dataset: customers
Raw run mode: FULL_LOAD
Normalized run mode: FULL_LOAD
Reprocess successful: True
Backfill start date: (not set)
Backfill end date: (not set)
Landing: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/
Bronze: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/customers/
Manifest: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/


In [0]:

# CELL 2 — DISCOVER LANDING FILES AND HASH CONTENT (READ-ONLY)

def list_files_recursive(path):
    found = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            found.extend(list_files_recursive(item.path))
        else:
            found.append((item.name, item.path, int(item.size)))

    return found


discovered = list_files_recursive(LANDING_PATH)

landing_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_size_bytes", LongType(), False),
])

landing_df = (
    spark.createDataFrame(discovered, landing_schema)
    if discovered
    else spark.createDataFrame([], landing_schema)
)

# Extract YYYYMMDD from filenames such as customers_20261007.json
dated_df = (
    landing_df
    .withColumn(
        "date_string",
        F.regexp_extract(
            F.col("file_name"),
            r"^customers_(\d{8})\.json$",
            1
        )
    )
    .withColumn(
        "data_date",
        F.expr("try_to_date(date_string, 'yyyyMMdd')")
    )
    .withColumn(
        "filename_status",
        F.when(F.col("date_string") == "", "INVALID_FILENAME")
         .when(F.col("data_date").isNull(), "INVALID_DATE")
         .otherwise("VALID")
    )
)

# Display invalid filenames/dates before stopping.
invalid_df = dated_df.filter(F.col("filename_status") != "VALID")

if invalid_df.limit(1).count() > 0:
    display(
        invalid_df.select(
            "file_name", "date_string", "filename_status"
        )
    )
    raise ValueError(
        "Invalid customer filenames or dates found. "
        "Review the displayed rows before processing."
    )

# Calculate SHA-256 hashes for discovered files.
binary_df = (
    spark.read.format("binaryFile")
    .load(LANDING_PATH + "**")
    .select(
        F.col("path").alias("file_path"),
        F.sha2(F.col("content"), 256).alias("actual_content_hash")
    )
)

files_df = (
    dated_df
    .filter(F.col("filename_status") == "VALID")
    .join(binary_df, on="file_path", how="left")
)

if files_df.filter(F.col("actual_content_hash").isNull()).limit(1).count() > 0:
    raise ValueError("Could not calculate a hash for every discovered file.")

display(
    files_df.select(
        "file_name",
        "file_path",
        "data_date",
        "file_size_bytes",
        "actual_content_hash"
    ).orderBy("data_date", "file_name")
)

print("Discovered customer files:", files_df.count())


file_name,file_path,data_date,file_size_bytes,actual_content_hash
customers_20261005.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=05/customers_20261005.json,2026-10-05,2170002,64154d3f4ec59d56ac99d68032542843640de32a3e1eaea141d9f5f9404bdefd
customers_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,2026-10-07,2908940,5ef2c76d178ce3e2b35c8413a647bf9f5183d6404ffca94308d02a371d9765e0
customers_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json,2026-10-08,2072549,d82dc9b7776651362b54e203ec0734ae3a495534b7f093c9e2302e64033b5ef5
customers_20261009.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json,2026-10-09,2735503,31246e22f0a2e56887ab66100a0c69832b3c77532d3a3197a1d80f6430a3b724
customers_20261012.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=12/customers_20261012.json,2026-10-12,331,e15b6e5b19e0af1c49a8f884df23d84213be4feca76cdf4f73a08996c8f1661d


Discovered customer files: 5


In [0]:
# CELL 3 — MANIFEST RECONCILIATION AND RUN-MODE SELECTION (READ-ONLY)
if not DeltaTable.isDeltaTable(spark, MANIFEST_PATH):
    raise RuntimeError(f"Global manifest Delta table not found: {MANIFEST_PATH}")

manifest_all_df = spark.read.format("delta").load(MANIFEST_PATH)
required_manifest_columns = {
    "dataset", "file_name", "file_path", "status", "content_hash",
    "data_date", "batch_id", "run_id", "retry_count", "attempt_count",
    "claimed_at", "lease_expires_at", "last_attempt_at", "status_updated_at",
    "error_message", "error_type", "last_error_at", "processed_at",
    "last_success_at", "processed_content_hash", "source_record_count",
    "written_record_count", "rejected_record_count", "target_path",
    "target_delta_version", "file_size_bytes", "first_seen", "last_seen_at",
    "discovery_run_id"
}
missing = sorted(required_manifest_columns - set(manifest_all_df.columns))
if missing:
    raise ValueError(f"Global manifest missing required columns: {missing}")

manifest_df = (
    manifest_all_df.filter(F.col("dataset") == DATASET_NAME)
    .select("dataset", "file_name", "file_path", "status", "content_hash",
            "data_date", "batch_id", "run_id", "retry_count", "attempt_count")
)
duplicate_keys = manifest_df.groupBy("dataset", "file_name").count().filter("count > 1").count()
assert duplicate_keys == 0, "Duplicate dataset/file_name keys in manifest."

reconciliation_df = (
    files_df.alias("l")
    .join(manifest_df.alias("m"), F.col("l.file_name") == F.col("m.file_name"), "left")
    .select(
        F.col("l.file_name"), F.col("l.file_path"), F.col("l.file_size_bytes"),
        F.col("l.data_date").alias("landing_data_date"), F.col("l.actual_content_hash"),
        F.col("m.status").alias("manifest_status"), F.col("m.content_hash").alias("manifest_hash"),
        F.col("m.data_date").alias("manifest_data_date"), F.col("m.file_path").alias("manifest_file_path")
    )
    .withColumn("effective_data_date", F.coalesce("manifest_data_date", "landing_data_date"))
    .withColumn(
        "selection_reason",
        F.when(F.col("manifest_status") == "PROCESSING", "SKIP_PROCESSING")
         .when(F.col("manifest_status").isNull(), "REGISTER_REQUIRED")
         .when(F.col("manifest_hash").isNull(), "REVIEW_REQUIRED")
         .when(F.col("manifest_hash") != F.col("actual_content_hash"), "CONTENT_CHANGED")
         .when(F.col("manifest_status").isin("NEW", "FAILED"), "ELIGIBLE")
         .when(F.col("manifest_status") == "SUCCESS", "SUCCESS_REPROCESS_CANDIDATE")
         .otherwise("REVIEW_REQUIRED")
    )
)

base_reasons = ["REGISTER_REQUIRED", "ELIGIBLE", "CONTENT_CHANGED"]
if RUN_MODE == "INCREMENTAL":
    selected_df = reconciliation_df.filter(F.col("selection_reason").isin(base_reasons))
elif RUN_MODE == "BACKFILL":
    reasons = base_reasons + (["SUCCESS_REPROCESS_CANDIDATE"] if REPROCESS_SUCCESSFUL else [])
    selected_df = reconciliation_df.filter(
        F.col("effective_data_date").between(
            F.to_date(F.lit(BACKFILL_START_DATE)), F.to_date(F.lit(BACKFILL_END_DATE))
        ) & F.col("selection_reason").isin(reasons)
    )
else:
    selected_df = reconciliation_df.filter(
        F.col("selection_reason").isin(base_reasons + ["SUCCESS_REPROCESS_CANDIDATE"])
    )

display(reconciliation_df.select("file_name", "effective_data_date", "manifest_status",
                                 "selection_reason").orderBy("effective_data_date", "file_name"))
print("Selected candidates:", selected_df.count())


file_name,effective_data_date,manifest_status,selection_reason
customers_20261005.json,2026-10-05,PROCESSING,SKIP_PROCESSING
customers_20261007.json,2026-10-07,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
customers_20261008.json,2026-10-08,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
customers_20261009.json,2026-10-09,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
customers_20261012.json,2026-10-12,SUCCESS,SUCCESS_REPROCESS_CANDIDATE


Selected candidates: 4


In [0]:
# CELL 4 — PREFLIGHT AND SCHEMA CHECKS (READ-ONLY)
selected_count = selected_df.count()
assert selected_df.groupBy("file_name").count().filter("count > 1").count() == 0, "Duplicate selected files."
assert selected_df.filter(F.col("actual_content_hash").isNull() | (F.length("actual_content_hash") != 64)).count() == 0, "Missing/invalid source hash."
assert selected_df.filter(F.col("effective_data_date").isNull()).count() == 0, "Missing data date."
assert selected_df.filter(F.col("manifest_status") == "PROCESSING").count() == 0, "A PROCESSING file was selected."
assert selected_df.filter(F.col("selection_reason") == "REVIEW_REQUIRED").count() == 0, "A review-required file was selected."

if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    existing_bronze_df = spark.read.format("delta").load(BRONZE_PATH)
    missing_bronze = sorted(set(BRONZE_COLUMNS) - set(existing_bronze_df.columns))
    extra_bronze = sorted(set(existing_bronze_df.columns) - set(BRONZE_COLUMNS))
    if missing_bronze or extra_bronze:
        raise ValueError(f"Existing Bronze columns mismatch. Missing={missing_bronze}; extra={extra_bronze}")
else:
    print("Bronze Delta target does not exist yet; it will be created on first write.")

display(selected_df.select("file_name", "file_path", "effective_data_date",
                           "manifest_status", "selection_reason", "actual_content_hash"))
print("Preflight PASS | selected:", selected_count)


file_name,file_path,effective_data_date,manifest_status,selection_reason,actual_content_hash
customers_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,2026-10-07,SUCCESS,SUCCESS_REPROCESS_CANDIDATE,5ef2c76d178ce3e2b35c8413a647bf9f5183d6404ffca94308d02a371d9765e0
customers_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json,2026-10-08,SUCCESS,SUCCESS_REPROCESS_CANDIDATE,d82dc9b7776651362b54e203ec0734ae3a495534b7f093c9e2302e64033b5ef5
customers_20261009.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json,2026-10-09,SUCCESS,SUCCESS_REPROCESS_CANDIDATE,31246e22f0a2e56887ab66100a0c69832b3c77532d3a3197a1d80f6430a3b724
customers_20261012.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=12/customers_20261012.json,2026-10-12,SUCCESS,SUCCESS_REPROCESS_CANDIDATE,e15b6e5b19e0af1c49a8f884df23d84213be4feca76cdf4f73a08996c8f1661d


Preflight PASS | selected: 4


In [0]:

# CELL 5 — PROCESS SELECTED FILES
# This cell performs manifest and Bronze writes. Use ONE writer for this dataset.

run_id = str(uuid.uuid4())

manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)
manifest_schema = spark.read.format("delta").load(MANIFEST_PATH).schema
manifest_columns = [f.name for f in manifest_schema.fields]

results = []

print("Processing run ID:", run_id)


def register_manifest_file(file_name, file_path, file_size, data_date, content_hash):
    now = datetime.now(timezone.utc).replace(tzinfo=None)

    values = {name: None for name in manifest_columns}
    values.update({
        "dataset": DATASET_NAME,
        "file_name": file_name,
        "file_path": file_path,
        "status": "NEW",
        "first_seen": now,
        "batch_id": str(uuid.uuid4()),
        "retry_count": 0,
        "data_date": data_date,
        "file_size_bytes": int(file_size),
        "content_hash": content_hash,
        "last_seen_at": now,
        "discovery_run_id": run_id,
        "attempt_count": 0,
        "status_updated_at": now
    })

    new_row = spark.createDataFrame([values], schema=manifest_schema)

    (
        manifest_delta.alias("t")
        .merge(
            new_row.alias("s"),
            "t.dataset = s.dataset AND t.file_name = s.file_name"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


for c in selected_df.orderBy("effective_data_date", "file_name").collect():
    file_name = c["file_name"]
    file_path = c["file_path"]
    actual_hash = c["actual_content_hash"]
    reason = c["selection_reason"]

    bronze_committed = False
    claimed = False
    source_count = None
    written_count = None
    delta_version = None

    try:
        # 1. Verify the file has not changed since discovery.
        current_hash = (
            spark.read.format("binaryFile")
            .load(file_path)
            .select(F.sha2("content", 256).alias("h"))
            .first()["h"]
        )

        if current_hash != actual_hash:
            raise RuntimeError(
                f"{file_name}: source changed after discovery."
            )

        # 2. Read the current manifest entry.
        rows = (
            spark.read.format("delta").load(MANIFEST_PATH)
            .filter(
                (F.col("dataset") == DATASET_NAME) &
                (F.col("file_name") == file_name)
            )
            .select("status", "content_hash", "file_path")
            .collect()
        )

        # 3. Register a genuinely unregistered file.
        if not rows:
            if reason != "REGISTER_REQUIRED":
                raise RuntimeError(
                    f"{file_name}: manifest row disappeared; rerun discovery."
                )

            register_manifest_file(
                file_name,
                file_path,
                c["file_size_bytes"],
                c["effective_data_date"],
                actual_hash
            )

            rows = (
                spark.read.format("delta").load(MANIFEST_PATH)
                .filter(
                    (F.col("dataset") == DATASET_NAME) &
                    (F.col("file_name") == file_name)
                )
                .select("status", "content_hash", "file_path")
                .collect()
            )

        if len(rows) != 1:
            raise RuntimeError(
                f"{file_name}: expected exactly one manifest row."
            )

        old = rows[0]

        if old["status"] == "PROCESSING":
            raise RuntimeError(f"{file_name}: already PROCESSING.")

        if old["file_path"] and old["file_path"] != file_path:
            raise RuntimeError(
                f"{file_name}: manifest file_path differs from discovered path."
            )

        if reason == "CONTENT_CHANGED":
            if old["content_hash"] == actual_hash:
                raise RuntimeError(
                    f"{file_name}: content no longer appears changed; rerun discovery."
                )
        elif old["content_hash"] != actual_hash:
            raise RuntimeError(
                f"{file_name}: manifest hash mismatch; rerun discovery."
            )

        if (
            old["status"] == "SUCCESS"
            and RUN_MODE == "INCREMENTAL"
            and reason != "CONTENT_CHANGED"
        ):
            raise RuntimeError(
                f"{file_name}: successful file not eligible for incremental reprocessing."
            )

        if (
            old["status"] == "SUCCESS"
            and RUN_MODE == "BACKFILL"
            and not REPROCESS_SUCCESSFUL
        ):
            raise RuntimeError(
                f"{file_name}: successful-file reprocessing disabled for this backfill."
            )

        # 4. Claim the manifest row with a lease.
        expected_status = old["status"]
        expected_hash = old["content_hash"]

        now = datetime.now(timezone.utc).replace(tzinfo=None)
        lease_expiry = now + timedelta(minutes=LEASE_MINUTES)

        manifest_delta.update(
            condition=(
                (F.col("dataset") == DATASET_NAME) &
                (F.col("file_name") == file_name) &
                (F.col("status") == expected_status) &
                (F.col("content_hash") == expected_hash)
            ),
            set={
                "status": F.lit("PROCESSING"),
                "run_id": F.lit(run_id),
                "claimed_at": F.lit(now).cast("timestamp"),
                "lease_expires_at": F.lit(lease_expiry).cast("timestamp"),
                "last_attempt_at": F.lit(now).cast("timestamp"),
                "attempt_count": F.coalesce(
                    F.col("attempt_count"), F.lit(0)
                ) + 1,
                "status_updated_at": F.lit(now).cast("timestamp"),
                "content_hash": F.lit(actual_hash),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string")
            }
        )

        owner = (
            spark.read.format("delta").load(MANIFEST_PATH)
            .filter(
                (F.col("dataset") == DATASET_NAME) &
                (F.col("file_name") == file_name)
            )
            .select("status", "run_id", "content_hash")
            .first()
        )

        if (
            not owner
            or owner["status"] != "PROCESSING"
            or owner["run_id"] != run_id
            or owner["content_hash"] != actual_hash
        ):
            raise RuntimeError(
                f"{file_name}: manifest claim could not be verified."
            )

        claimed = True

        # 5. Read and validate the source.
        source_df = (
            spark.read.schema(SOURCE_SCHEMA)
            .option("multiLine", "true")
            .option("mode", "FAILFAST")
            .json(file_path)
        )

        source_count = source_df.count()

        if source_count == 0:
            raise ValueError(f"{file_name}: source file is empty.")

        null_ids = source_df.filter(
            F.col("customer_id").isNull()
        ).count()

        if null_ids:
            raise ValueError(
                f"{file_name}: null customer_id values found."
            )

        # Check the 32-bit integer range before casting.
        id_range = source_df.agg(
            F.min("customer_id").alias("min_id"),
            F.max("customer_id").alias("max_id")
        ).first()

        min_id = id_range["min_id"]
        max_id = id_range["max_id"]

        if (
            min_id is not None and min_id < -2147483648
        ) or (
            max_id is not None and max_id > 2147483647
        ):
            raise ValueError(
                f"{file_name}: customer_id is outside the 32-bit integer range. "
                f"min_id={min_id}, max_id={max_id}. "
                "Bronze schema migration is required; no Bronze write was performed."
            )

        # 6. Prepare rows to match the existing Bronze schema.
        # Existing Bronze customer_id type is INT, not BIGINT.
        batch_rows = (
            spark.read.format("delta").load(MANIFEST_PATH)
            .filter(
                (F.col("dataset") == DATASET_NAME) &
                (F.col("file_name") == file_name)
            )
            .select("batch_id")
            .collect()
        )

        if len(batch_rows) != 1 or not batch_rows[0]["batch_id"]:
            raise RuntimeError(
                f"{file_name}: missing or duplicate manifest batch_id."
            )

        batch_id = batch_rows[0]["batch_id"]

        prepared_df = (
            source_df
            .withColumn("customer_id", F.col("customer_id").cast("int"))
            .withColumn("name", F.col("name").cast("string"))
            .withColumn("city", F.col("city").cast("string"))
            .withColumn("market", F.col("market").cast("string"))
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("source_system", F.lit(SOURCE_SYSTEM))
            .withColumn("source_file", F.lit(file_path))
            .withColumn("run_id", F.lit(run_id))
            .withColumn("batch_id", F.lit(batch_id))
            .select(*BRONZE_COLUMNS)
        )

        # 7. Check exact schema compatibility before writing.
        if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
            target_df = spark.read.format("delta").load(BRONZE_PATH)

            expected = [
                (f.name, f.dataType.simpleString())
                for f in prepared_df.schema
            ]
            actual = [
                (f.name, f.dataType.simpleString())
                for f in target_df.schema
            ]

            if expected != actual:
                raise RuntimeError(
                    f"{file_name}: Bronze schema mismatch. "
                    f"Prepared={expected}; target={actual}"
                )

        # 8. Replace this source file only; preserve other files.
        path_literal = file_path.replace("'", "''")

        (
            prepared_df.write.format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"source_file = '{path_literal}'")
            .save(BRONZE_PATH)
        )

        bronze_committed = True

        # 9. Verify the committed Bronze data.
        post_df = (
            spark.read.format("delta").load(BRONZE_PATH)
            .filter(F.col("source_file") == file_path)
        )

        written_count = post_df.count()

        if written_count != source_count:
            raise RuntimeError(
                f"{file_name}: source/written count mismatch. "
                f"source={source_count}, written={written_count}"
            )

        if post_df.filter(F.col("customer_id").isNull()).count() != 0:
            raise RuntimeError(
                f"{file_name}: null customer_id after write."
            )

        delta_version = (
            DeltaTable.forPath(spark, BRONZE_PATH)
            .history(1)
            .select("version")
            .first()["version"]
        )

        # 10. Finalize the manifest only if this run still owns the claim.
        finished = datetime.now(timezone.utc).replace(tzinfo=None)

        DeltaTable.forPath(spark, MANIFEST_PATH).alias("t").update(
            condition=(
                (F.col("t.dataset") == DATASET_NAME) &
                (F.col("t.file_name") == file_name) &
                (F.col("t.status") == "PROCESSING") &
                (F.col("t.run_id") == run_id) &
                (F.col("t.content_hash") == actual_hash)
            ),
            set={
                "status": F.lit("SUCCESS"),
                "processed_at": F.lit(finished).cast("timestamp"),
                "last_success_at": F.lit(finished).cast("timestamp"),
                "status_updated_at": F.lit(finished).cast("timestamp"),
                "processed_content_hash": F.lit(actual_hash),
                "source_record_count": F.lit(source_count).cast("long"),
                "written_record_count": F.lit(written_count).cast("long"),
                "rejected_record_count": F.lit(0).cast("long"),
                "target_path": F.lit(BRONZE_PATH),
                "target_delta_version": F.lit(delta_version).cast("long"),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string"),
                "last_error_at": F.lit(None).cast("timestamp"),
                "claimed_at": F.lit(None).cast("timestamp"),
                "lease_expires_at": F.lit(None).cast("timestamp")
            }
        )

        final = (
            spark.read.format("delta").load(MANIFEST_PATH)
            .filter(
                (F.col("dataset") == DATASET_NAME) &
                (F.col("file_name") == file_name)
            )
            .select("status", "run_id", "written_record_count")
            .first()
        )

        if (
            not final
            or final["status"] != "SUCCESS"
            or final["run_id"] != run_id
            or final["written_record_count"] != written_count
        ):
            raise RuntimeError(
                f"{file_name}: final manifest verification failed."
            )

        results.append(
            (file_name, "SUCCESS", source_count, written_count, delta_version, "")
        )

        print(
            f"SUCCESS {file_name}: source={source_count}, "
            f"written={written_count}, Delta version={delta_version}"
        )

    except Exception as exc:
        error_text = f"{type(exc).__name__}: {str(exc)}"[:4000]

        if claimed:
            try:
                current_owner = (
                    spark.read.format("delta").load(MANIFEST_PATH)
                    .filter(
                        (F.col("dataset") == DATASET_NAME) &
                        (F.col("file_name") == file_name)
                    )
                    .select("status", "run_id")
                    .first()
                )

                if (
                    current_owner
                    and current_owner["status"] == "PROCESSING"
                    and current_owner["run_id"] == run_id
                ):
                    if bronze_committed:
                        error_text = (
                            "BRONZE_WRITE_COMMITTED_BUT_FINALIZATION_FAILED; "
                            "reconcile before retry. " + error_text
                        )[:4000]

                    DeltaTable.forPath(spark, MANIFEST_PATH).alias("t").update(
                        condition=(
                            (F.col("t.dataset") == DATASET_NAME) &
                            (F.col("t.file_name") == file_name) &
                            (F.col("t.status") == "PROCESSING") &
                            (F.col("t.run_id") == run_id)
                        ),
                        set={
                            "status": F.lit("FAILED"),
                            "status_updated_at": F.lit(
                                datetime.now(timezone.utc).replace(tzinfo=None)
                            ).cast("timestamp"),
                            "last_error_at": F.current_timestamp(),
                            "error_message": F.lit(error_text),
                            "error_type": F.lit(type(exc).__name__),
                            "claimed_at": F.lit(None).cast("timestamp"),
                            "lease_expires_at": F.lit(None).cast("timestamp")
                        }
                    )

            except Exception as update_exc:
                print(
                    "Could not safely update manifest failure state:",
                    str(update_exc)[:1000]
                )

        status = (
            "FAILED_AFTER_BRONZE_WRITE_RECONCILIATION_REQUIRED"
            if bronze_committed else "FAILED"
        )

        print(f"{status} {file_name}: {error_text}")

        results.append(
            (file_name, status, source_count, written_count, delta_version, error_text)
        )


results_df = spark.createDataFrame(
    results,
    """
    file_name string,
    status string,
    source_count long,
    written_count long,
    delta_version long,
    error_message string
    """
)

display(results_df)
print("Run ID:", run_id)


Processing run ID: 03e6dfd8-a9cf-4d86-b124-51b3db1d5a24
SUCCESS customers_20261007.json: source=25000, written=25000, Delta version=20
SUCCESS customers_20261008.json: source=25000, written=25000, Delta version=21
SUCCESS customers_20261009.json: source=25000, written=25000, Delta version=22
SUCCESS customers_20261012.json: source=3, written=3, Delta version=23


file_name,status,source_count,written_count,delta_version,error_message
customers_20261007.json,SUCCESS,25000,25000,20,
customers_20261008.json,SUCCESS,25000,25000,21,
customers_20261009.json,SUCCESS,25000,25000,22,
customers_20261012.json,SUCCESS,3,3,23,


Run ID: 03e6dfd8-a9cf-4d86-b124-51b3db1d5a24


In [0]:
# CELL 6 — FINAL BRONZE INVENTORY AND MANIFEST STATUS (READ-ONLY)
bronze_final_df = spark.read.format("delta").load(BRONZE_PATH)
manifest_final_df = spark.read.format("delta").load(MANIFEST_PATH)

display(
    bronze_final_df.filter(F.col("source_system") == SOURCE_SYSTEM)
    .groupBy("source_file")
    .agg(
        F.count("*").alias("bronze_rows"),
        F.countDistinct("customer_id").alias("distinct_customer_ids"),
        F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_ids")
    )
    .orderBy("source_file")
)

display(
    manifest_final_df.filter(F.col("dataset") == DATASET_NAME)
    .select("file_name", "status", "run_id", "source_record_count",
            "written_record_count", "target_delta_version", "lease_expires_at", "error_message")
    .orderBy("file_name")
)
print("Total customers Bronze rows:", bronze_final_df.filter(F.col("source_system") == SOURCE_SYSTEM).count())


source_file,bronze_rows,distinct_customer_ids,null_customer_ids
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=09/customers_20261009.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=12/customers_20261012.json,3,3,0


file_name,status,run_id,source_record_count,written_record_count,target_delta_version,lease_expires_at,error_message
customers_20261005.json,PROCESSING,3a4edd77-6a74-48cd-b6ab-7fed9c4b502c,null,null,null,2026-10-12T12:42:32.338Z,null
customers_20261007.json,SUCCESS,03e6dfd8-a9cf-4d86-b124-51b3db1d5a24,25000,25000,20,null,null
customers_20261008.json,SUCCESS,03e6dfd8-a9cf-4d86-b124-51b3db1d5a24,25000,25000,21,null,null
customers_20261009.json,SUCCESS,03e6dfd8-a9cf-4d86-b124-51b3db1d5a24,25000,25000,22,null,null
customers_20261012.json,SUCCESS,03e6dfd8-a9cf-4d86-b124-51b3db1d5a24,3,3,23,null,null


Total customers Bronze rows: 75003


In [0]:
from pyspark.sql import functions as F

display(
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME) &
        (F.col("file_name") == "customers_20261005.json")
    )
    .select(
        "file_name",
        "status",
        "run_id",
        "source_record_count",
        "written_record_count",
        "target_delta_version",
        "lease_expires_at",
        "error_message"
    )
)

file_name,status,run_id,source_record_count,written_record_count,target_delta_version,lease_expires_at,error_message
customers_20261005.json,PROCESSING,3a4edd77-6a74-48cd-b6ab-7fed9c4b502c,null,null,null,2026-10-12T12:42:32.338Z,null
